### 0. Initial Setup: Drive & Dependencies
Before starting the pipeline, we mount Google Drive to access the project files and install the necessary libraries.

In [1]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define and change to the project root directory
PROJECT_ROOT_PATH = '/content/drive/MyDrive/Colab Notebooks/Senior Project'
os.chdir(PROJECT_ROOT_PATH)

print(f"Current working directory: {os.getcwd()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Current working directory: /content/drive/MyDrive/Colab Notebooks/Senior Project


In [2]:
# Install dependencies from requirements.txt
if os.path.exists('requirements.txt'):
    !pip install -r requirements.txt
else:
    print("requirements.txt not found in the current directory.")

In [3]:
!pip install "https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.11/flash_attn-2.8.3+cu12torch2.11cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.7/253.7 MB 9.8 MB/s eta 0:00:00


# Omni-Phi: End-to-End Speech-to-Speech Translation Pipeline

This notebook demonstrates the end-to-end training and inference workflow for the **Omni-Phi** S2ST model. Omni-Phi translates spoken English to spoken German directly, bypasses intermediate text decoding, and leverages the frozen EnCodec codes combined with the **microsoft/Phi-4-multimodal-instruct** model's audio capabilities via a speech LoRA adapter.

### Pipeline Stages:
1. **Environment Setup & Path Configuration**
2. **Phase 1: Run Target Tokenization & Preprocessing**
3. **Phase 2 & 3: Load Multimodal Model, Processor, & Dataset**
4. **Phase 4: Run Training Loop using Custom Data Collator**
5. **Phase 5: Run Speech-to-Speech Translation Inference**

## 1. Environment Setup & Path Configuration

First, we import core dependencies and add the project root to `sys.path` so we can import our modules (`dataset`, `model`, `collator`, `inference`, `encoders`, etc.).

In [4]:
import os
import sys
from pathlib import Path

# Resolve directories relative to the current notebook location
notebook_dir = Path(os.getcwd()).resolve()
project_root = notebook_dir
omni_phi_dir = project_root / "models" / "omni_phi"

# Add paths to sys.path
# We add both the root and the omni_phi subdirectory to ensure all local modules are findable
paths_to_add = [str(project_root), str(omni_phi_dir)]
for path in paths_to_add:
    if path not in sys.path:
        sys.path.append(path)

print(f"Project root  : {project_root}")
print(f"Omni Phi path : {omni_phi_dir}")
print(f"Active paths in sys.path: {[p for p in sys.path if 'Senior Project' in p]}")

Project root  : /content/drive/MyDrive/Colab Notebooks/Senior Project
Omni Phi path : /content/drive/MyDrive/Colab Notebooks/Senior Project/models/omni_phi
Active paths in sys.path: ['/content/drive/MyDrive/Colab Notebooks/Senior Project', '/content/drive/MyDrive/Colab Notebooks/Senior Project/models/omni_phi']


## 2. Phase 1: Preprocessing & Tokenization

Before training, we must extract source waveforms and encode target audios into interleaved EnCodec codebook tokens offset by `+100,000` to fit the Phi-4 vocabulary.

We can trigger Phase 1 preprocessing directly from the notebook.

> **Note:** For demonstration purposes, we will preprocess a small subset of the `fleurs` dataset. Adjust `--num_samples` as needed for full training.

In [ ]:
# Run Phase 1 Preprocessing script
!python "{project_root}/models/omni_phi/preprocess_omni.py" \
    --dataset fleurs \
    --split train \
    --num_samples 15000 \
    --bandwidth 1.5 \
    --token_offset 100000

## 3. Phase 2 & 3: Load Model, Processor, & Dataset

Now we import our custom dataset class (`OmniPhiDataset`) and core model wrapper (`OmniPhiS2ST`). We will load the underlying multimodal processor, prepare datasets from preprocessed JSONL files, and load our wrapper with the LoRA speech adapter active.

In [5]:
import torch
from transformers import AutoProcessor
from dataset import OmniPhiDataset
from model import OmniPhiS2ST

checkpoint_output_dir = omni_phi_dir / "checkpoints"
PHI4_HUB_ID = "microsoft/Phi-4-multimodal-instruct"

# MODEL_ID = "microsoft/Phi-4-multimodal-instruct"    # Start from the fresh weights
MODEL_ID = str(checkpoint_output_dir)                 # Resume from last checkpoint
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# 1. Load Processor
print("Loading AutoProcessor...")
processor = AutoProcessor.from_pretrained(PHI4_HUB_ID, trust_remote_code=True)

# 2. Load custom training dataset and evaluation dataset
data_path = omni_phi_dir / "data" / "preprocessed"
train_jsonl = data_path / "train.jsonl"

print(f"Loading dataset from {train_jsonl}...")
train_dataset = OmniPhiDataset(str(train_jsonl), processor, training=True)

# 3. Load Omni-Phi Model Wrapper (with frozen EnCodec and trainable LoRA speech adapter)
print("Loading model wrapper...")
model = OmniPhiS2ST(phi4_model_id=MODEL_ID, device=device)
model.processor = processor


Device: cuda
Loading AutoProcessor...


/usr/local/lib/python3.12/dist-packages/transformers/models/auto/image_processing_auto.py:647: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading dataset from /content/drive/MyDrive/Colab Notebooks/Senior Project/models/omni_phi/data/preprocessed/train.jsonl...
[OmniPhiDataset] Cache hit — 15000 samples found in /content/drive/MyDrive/Colab Notebooks/Senior Project/models/omni_phi/data/preprocessed/.cache_train. Skipping JSONL scan.
[OmniPhiDataset] Cache already complete (15000 samples). Skipping.
Loading model wrapper...
[OmniPhiS2ST] Loading AutoProcessor...
[OmniPhiS2ST] Processor load failed (TypeError). Falling back to hub: microsoft/Phi-4-multimodal-instruct


`torch_dtype` is deprecated! Use `dtype` instead!


[OmniPhiS2ST] Loading Phi-4 model...
[OmniPhiS2ST] FlashAttention-2 detected — using flash_attention_2.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

[OmniPhiS2ST] LoRA speech adapter applied. Non-audio layers frozen.
[OmniPhiS2ST] Loading VQGANEncoder (EnCodec)...
[OmniPhiS2ST] VQGANEncoder frozen on cuda.


## 4. Phase 4: Data Collator & Training Loop

With our custom `omni_phi_collate_fn` import, we pad multimodal batch sequences leftwards while masking the prompt region (`labels = -100`), ensuring the model learns exclusively to predict target interleaved EnCodec token sequences.

In [7]:
import torch
from collator import omni_phi_collate_fn
from trainer import OmniPhiTrainer
from transformers import TrainingArguments

# =========================================================================
# PHASE 1: PREPARE MODEL & SELECT ADAPTER
# =========================================================================
model.phi4.set_lora_adapter('speech')

# Enforce freezing and trainable parameters strictly BEFORE trainer initialization
lora_count = 0
for name, param in model.phi4.named_parameters():
    if 'lora' in name.lower() or 'embed_tokens' in name.lower() or 'lm_head' in name.lower():
        param.requires_grad_(True)
        if 'lora' in name.lower():
            lora_count += 1
    else:
        param.requires_grad_(False)

# Force base model input require grads hook (which works in tandem with our model.py wrapper fix!)
model.phi4.enable_input_require_grads()
model.phi4.config.use_cache = False

# Quick verification checks
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ {lora_count} LoRA tensors marked trainable ({trainable_params:,} params)")
assert trainable_params > 0, "STOP: still no trainable params — do not proceed"

# =========================================================================
# PHASE 2: CONFIGURE ROBUST TRAINING ARGUMENTS
# =========================================================================
# Enable TF32 matrix calculations for speed boosts on NVIDIA A100 GPU
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

_optim = "adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch"

training_args = TrainingArguments(
    output_dir                    = str(checkpoint_output_dir),
    num_train_epochs              = 2,
    per_device_train_batch_size   = 4,            # keep — VRAM constrained
    gradient_accumulation_steps   = 4,            # effective batch 16
    gradient_checkpointing        = True,         # Kept active for optimization & VRAM safety
    gradient_checkpointing_kwargs = {"use_reentrant": False}, # Set to non-reentrant for stable custom pipelines

    # Optimizer
    optim         = _optim,
    adam_beta1    = 0.9,
    adam_beta2    = 0.95,
    adam_epsilon  = 1e-7,
    learning_rate = 4e-5,
    weight_decay  = 0.01,
    max_grad_norm = 5.0,

    # Scheduler
    lr_scheduler_type = "constant_with_warmup",
    warmup_ratio      = 0.01,

    # Precision
    bf16  = True,
    tf32  = True,
    fp16  = False,

    # Logging & saving
    logging_steps  = 10,
    save_strategy  = "no",
    save_safetensors = False,

    # DataLoader Settings
    dataloader_num_workers  = 4,
    dataloader_pin_memory   = True,

    remove_unused_columns    = False,
    report_to                = "none",
    ddp_find_unused_parameters = True,
)

# =========================================================================
# PHASE 3: INIT TRAINER & START TRAINING
# a
# The trainer now inspects the fully pre-configured model correctly
trainer = OmniPhiTrainer(
    model=model,
    args=training_args,
    data_collator=omni_phi_collate_fn,
    train_dataset=train_dataset,
)

print("Starting training...")
trainer.train()

# Save model weights
trainer.save_model(str(checkpoint_output_dir))

# Save processor/tokenizer components
try:
    model.processor.save_pretrained(str(checkpoint_output_dir))
except AttributeError:
    model.processor.tokenizer.save_pretrained(str(checkpoint_output_dir))
    print("Tokenizer saved (processor.save_pretrained skipped due to Phi4MM bug).")

print(f"✅ Training complete. Checkpoint saved to {checkpoint_output_dir}")


The model is already on multiple devices. Skipping the move to device specified in `args`.


✅ 512 LoRA tensors marked trainable (2,353,035,072 params)
Starting training...


Step,Training Loss
10,15.975600
20,16.025500
30,15.657500
40,15.580300
50,15.763300
60,15.762500
70,15.766200
80,15.682000
90,15.324200
100,15.711700


KeyboardInterrupt: 

In [9]:
try:
    model.processor.save_pretrained(str(checkpoint_output_dir))
except AttributeError:
    model.processor.tokenizer.save_pretrained(str(checkpoint_output_dir))
    print("✅ Tokenizer saved successfully.")


✅ Tokenizer saved successfully.


## 5. Phase 5: Inference

Now that the model is loaded and fine-tuned, we can translate spoken English audio to spoken German using both single-pass generation (`translate_speech`) and stream-friendly queue-based generation (`translate_speech_batched`).

In [6]:
import soundfile as sf
import numpy as np
from inference import translate_speech, translate_speech_batched
from dataset_loader import load_data, save_audio

# ── Load English sample from FLEURS test split ───────────────────────────
print("Loading English sample from FLEURS test split...")
datasets = load_data(
    dataset=["fleurs"],
    lang=["en"],
    split="train",
    num_samples=100,
)
en_ds = datasets.get("en")
if en_ds and len(en_ds) > 0:
    sample = en_ds[0]
    print(f"Loaded sample ID: {sample['id']}")
    print(f"Transcription: {sample['transcription']}")
    print(f"Gender: {sample['gender']}")

    # Save the sample audio to 'test_english.wav' for downstream inference steps
    source_wav = "test_english.wav"
    save_audio(sample, source_wav)
    print(f"Saved source audio to: {source_wav}")
else:
    source_wav = "test_english.wav"
    print(f"Warning: FLEURS test split sample not loaded. Expecting '{source_wav}'...")

# ── Option A: Single-Pass S2ST ─────────────────────────────────────────
# This translates an English wav file end-to-end in a single LLM forward-generation pass.
output_wav = "test_translated_german.wav"

if os.path.exists(source_wav):
    print(f"Running single-pass translation on {source_wav}...")
    translate_speech(source_wav_path=source_wav, output_wav_path=output_wav, model=model, max_new_tokens=450)
else:
    print(f"Please upload a '{source_wav}' file to run the translation!")

Loading English sample from FLEURS test split...
Loading google/fleurs (en_us) from local storage...
Validating en (audio & uniqueness)...
Final Count: 1476 common valid samples.
Loaded sample ID: 1
Transcription: on monday scientists from the stanford university school of medicine announced the invention of a new diagnostic tool that can sort cells by type a tiny printable chip that can be manufactured using standard inkjet printers for possibly about one u.s. cent each
Gender: 0
Saved source audio to: test_english.wav
Running single-pass translation on test_english.wav...
[inference] Translating test_english.wav (sampling rate: 16000 Hz)...
[generate_speech] Generating up to 450 tokens on cuda...


/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


TypeError: bad operand type for unary -: 'NoneType'

In [11]:
# ── Option B: Queue-Based Batched Inference ────────────────────────────
# Generates audio tokens progressively in small budgets (200 tokens each)
# and queues them together, allowing streaming-style translations.
if os.path.exists(source_wav):
    print(f"Running batched token-queue translation on {source_wav}...")
    audio_data, sr = sf.read(source_wav)
    if audio_data.ndim > 1:
        audio_data = audio_data.mean(axis=1)
    audio_data = audio_data.astype(np.float32)

    # Generate batched waveform
    translated_waveform = translate_speech_batched(model=model, source_audio=audio_data, source_sr=sr)

    # Save the output
    batched_output_path = "test_translated_german_batched.wav"
    sf.write(batched_output_path, translated_waveform, samplerate=24000)
    print(f"Batched translation completed successfully! Saved to: {batched_output_path}")

Running batched token-queue translation on test_english.wav...
[inference] Starting batched inference (token limit: 200/chunk, max 10 chunks)...
[inference] Generating chunk 1/10...
[generate_speech] Generating up to 200 tokens on cuda...
[generate_speech] Generation done. Total ids shape: torch.Size([1, 459])
[inference] Generating chunk 2/10...
[generate_speech] Generating up to 200 tokens on cuda...
[generate_speech] Generation done. Total ids shape: torch.Size([1, 459])
[inference] Generating chunk 3/10...
[generate_speech] Generating up to 200 tokens on cuda...
[generate_speech] Generation done. Total ids shape: torch.Size([1, 459])
[inference] Generating chunk 4/10...
[generate_speech] Generating up to 200 tokens on cuda...
[generate_speech] Generation done. Total ids shape: torch.Size([1, 459])
[inference] Generating chunk 5/10...
[generate_speech] Generating up to 200 tokens on cuda...
[generate_speech] Generation done. Total ids shape: torch.Size([1, 459])
[inference] Generati

In [ ]:
from google.colab import runtime
# Terminate the runtime
runtime.unassign()